In [1]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from color_mapping import clean_color_column

# 1. Datei laden
file_path = os.path.expanduser("~/Desktop/regge.xlsx")
df = pd.read_excel(file_path)


# 2. Preis bereinigen
def clean_numeric(val):
    if pd.isna(val) or val == "":
        return np.nan
    # Entfernt Währungen, Tausenderpunkte und wandelt Komma in Punkt um
    s = (
        str(val)
        .replace("€", "")
        .replace("EUR", "")
        .replace(".", "")
        .replace(",", ".")
        .strip()
    )
    try:
        return float(s)
    except:
        return np.nan


df["Preis"] = df["Preis"].apply(clean_numeric)

# Farbe bereinigen, BEVOR Dummy-Variablen erstellt werden
df["Farbe"] = clean_color_column(df["Farbe"])


# 3. Kategorische Features encoden
# WICHTIG: Füge hier alle Text-Spalten hinzu, die du im Modell berücksichtigen willst
categorical_cols = ["Farbe"]
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=float)

# 4. Zielvariable definieren
y = df_final["Preis"]

# 5. X definieren: NUR numerische Spalten behalten und 'Preis' entfernen
# Das entfernt automatisch alle restlichen Text-Spalten (z.B. Namen oder IDs)
X = df_final.select_dtypes(include=[np.number]).drop(columns=["Preis"])

# 6. Fehlende Werte (NaNs) entfernen
# OLS kann nicht mit NaNs arbeiten
mask = y.notna() & X.notna().all(axis=1)
X = X[mask]
y = y[mask]

# Prüfen, ob noch Daten übrig sind
if len(X) == 0:
    print(
        "Fehler: Keine Daten übrig nach der Bereinigung (alle Zeilen enthalten NaNs)."
    )
else:
    # 7. Konstante hinzufügen und Modell rechnen
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()

    print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  Preis   R-squared:                       0.401
Model:                            OLS   Adj. R-squared:                  0.322
Method:                 Least Squares   F-statistic:                     5.077
Date:                Fri, 08 May 2026   Prob (F-statistic):           0.000185
Time:                        13:22:35   Log-Likelihood:                -327.16
No. Observations:                  61   AIC:                             670.3
Df Residuals:                      53   BIC:                             687.2
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Batteriezustand   